In [ ]:
"""
import numpy as np
import importlib_metadata

print(importlib_metadata.version("numpy"))
if(importlib_metadata.version("numpy") !="1.26.4"):

  !pip uninstall numpy -y
  !pip install numpy==1.26.4
  import os
  os._exit(00)
"""

# Importe und Google Drive Setup
Der erste Chunk installiert und importiert die erforderlichen Module und richtet die Google Drive-Umgebung ein.

In [ ]:
"""
from google.colab import drive
import pickle
from os import listdir
import pandas as pd
import matplotlib.pyplot as plt
import os
import math
drive.mount('/content/drive')

!pip install -U brightway25
import bw2analyzer as ba
import bw2calc as bc
import bw2data as bd
import bw2io as bi
"""

In [ ]:

"""
ei_version = "3.11"

bd.projects.set_current("kregi_workshop")
ei = bi.import_ecoinvent_release(
    version=ei_version,
    system_model="cutoff",
    username="XXX",
    password="XXX")
os.environ["HOME"] = "/content/drive/Shareddrives/KREGI_Workshop/brightway/daten/DB"
bi.backup_project_directory("kregi_workshop")
bd.projects.delete_project("kregi_workshop")
"""

'\nbd.projects.set_current("kregi_workshop")\nei = bi.import_ecoinvent_release(\n    version=ei_version,\n    system_model="cutoff",\n    username="XXX",\n    password="XXX")\nos.environ["HOME"] = "/content/drive/Shareddrives/KREGI_Workshop/brightway/daten/DB"\nbi.backup_project_directory("kregi_workshop")\nbd.projects.delete_project("kregi_workshop")\n'

In [ ]:
"""
def list_files(directory):
    # Iterate over all files in the directory
    for dirpath, _, filenames in os.walk(directory):
        for filename in filenames:
            # Print the full path of each file
            return os.path.join(dirpath, filename)

path      = "/content/drive/MyDrive/DB/ecoinvent/bw_project/raw/"
path_detailed = list_files(path + ei_version)
bi.restore_project_directory(path_detailed, overwrite_existing=True)
bd.projects.set_current("ecoinvent_"+ ei_version)
"""

#cf_dir = '/content/drive/Shareddrives/PMF/CFs/characterization_factors_' + ei_version

Restoring project backup archive - this could take a few minutes...
Restored project: ecoinvent_3.11


In [ ]:
for db in list(bd.databases):
  if "cutoff" in db:
    ecoinvent_name = db

for db in list(bd.databases):
  if "biosphere" in db:
    biosphere_name = db


In [ ]:
################################
## Additional biosphere flows ##
################################

if len(bd.Database(biosphere_name).search("Overburden"))==0:
  overburden = bd.Database(biosphere_name).new_activity(**{
      'categories': ('natural resource', 'in ground'),
      'code'      : "overburden",
      'CAS number': None,
      'name'      : 'Overburden',
      'database'  : biosphere_name,
      'unit'      : 'kilogram',
      'type'      : 'natural resource'})
  overburden.save()


if len(bd.Database(biosphere_name).search("Biomass, used"))==0:
  biomass_used = bd.Database(biosphere_name).new_activity(**{
      'categories': ('natural resource', 'none'),
      'code'      : "biomass, used",
      'CAS number': None,
      'name'      : 'Biomass, used',
      'database'  : biosphere_name,
      'unit'      : 'kilogram',
      'type'      : 'natural resource'})
  biomass_used.save()


if len(bd.Database(biosphere_name).search("Biomass, unused"))==0:
  biomass_unused = bd.Database(biosphere_name).new_activity(**{
      'categories': ('natural resource', 'none'),
      'code'      : "biomass, unused",
      'CAS number': None,
      'name'      : 'Biomass, unused',
      'database'  : biosphere_name,
      'unit'      : 'kilogram',
      'type'      : 'natural resource'})
  biomass_unused.save()




overburden     = bd.Database(biosphere_name).search("Overburden")[0]
gangue         = bd.Database(biosphere_name).search("Gangue")[0]
biomass_used   = bd.Database(biosphere_name).search("Biomass, used")[0]
biomass_unused = bd.Database(biosphere_name).search("Biomass, unused")[0]


In [ ]:
############################
### O V E R B U R D E N  ###
############################

for act in bd.Database(ecoinvent_name):
  amount = 0
  needstobein = False
  alreadyin   = False
  if "market" not in act["name"] and "treatment" not in act["name"]:

    for ex in act.exchanges():
      if ex["name"] == "Overburden":
        alreadyin = True
      if ex["name"] == "non-sulfidic overburden, off-site" or ex["name"] == "spoil from hard coal mining" or ex["name"] == "spoil from lignite mining":
        amount = amount + abs(ex["amount"])
        needstobein = True

    if needstobein == True and alreadyin == True:

      for ex in act.exchanges():
        if ex["name"] == "Overburden":
          ex["amount"] == amount


    if needstobein == True and alreadyin == False:
      wanted_act = act
      new_exc = wanted_act.new_exchange(input= overburden, amount=amount, type = "biosphere")
      new_exc["name"] = "Overburden"
      new_exc.save()





In [ ]:
#################################
### B I O T I C    U S E D    ###
#################################

agriculture_categories_list = [
  "0111:Growing of cereals (except rice), leguminous crops and oil seeds",
  "0112:Growing of rice",
  "0113:Growing of vegetables and melons, roots and tubers",
  "0114:Growing of sugar cane",
  "0116:Growing of fibre crops",
  "0119:Growing of other non-perennial crops",

  "0121:Growing of grapes",
  "0122:Growing of tropical and subtropical fruits",
  "0123:Growing of citrus fruits",
  "0124:Growing of pome fruits and stone fruits",
  "0125:Growing of other tree and bush fruits and nuts",
  "0126:Growing of oleaginous fruits",
  "0127:Growing of beverage crops",
  "0128:Growing of spices, aromatic, drug and pharmaceutical crops",
  "0129:Growing of other perennial crops",
]

forestry_categories_list = [
  "0210:Silviculture and other forestry activities",
  "0220:Logging"
]

animal_categories_list = [

  "0141:Raising of cattle and buffaloes",
  "0144:Raising of sheep and goats",
  "0145:Raising of swine|pigs",
  "0146:Raising of poultry",
  "0149:Raising of other animals"

]

### Agriculture ###


### Fishing ###

for act in bd.Database(ecoinvent_name):
  skip        = True
  biomass_amount = 0

  for ex in act.exchanges():
    if "Fish," in ex["name"]:
      skip = False
      biomass_amount = biomass_amount + ex.amount

  if skip == False:
    wanted_act = act
    new_exc = wanted_act.new_exchange(input= biomass_used, amount=biomass_amount, type = "biosphere")
    new_exc["name"] = "Biomass, used"
    new_exc.save()


### Rest ###

for act in bd.Database(ecoinvent_name):
  skip = True
  mass_per_energy = 1 / 19.5
  for ex in act.exchanges():
    if ex["name"] =="Energy, gross calorific value, in biomass":
      skip = False
      biomass_amount = mass_per_energy * ex.amount

  for ex in act.exchanges():
    if ex["name"] == "Biomass, used":
      skip = True

  if skip == False:
    wanted_act = act
    new_exc = wanted_act.new_exchange(input= biomass_used, amount=biomass_amount, type = "biosphere")
    new_exc["name"] = "Biomass, used"
    new_exc.save()


In [ ]:
#####################################
### B I O T I C    U N U S E D    ###
#####################################

def mean_function(x):
  return sum(x)/len(x)

### Agriculture ###
mean_moisture_content_resdidue = 0.5

for act in bd.Database(ecoinvent_name):
  skip = True
  if "market" not in act["name"]:
    for cat in agriculture_categories_list:
      if cat in act["classifications"][0]:
        if act["unit"] == "kilogram":
          skip = False

  for ex in act.exchanges():
    if ex["name"] == "Biomass, unused":
      skip = True

  if skip == False:
    for ex in act.exchanges():
      residue_ratio = 1

      if "wheat" in act["name"]:
        residue_ratio = mean_function([1.3,1.2,1.34,1.75,0.6,1,
                                      1.7,1.7,1.6,0.8,1.7,
                                      1.3,1.3,1.5,0.9,1.3])
      if "barley" in act["name"]:
        residue_ratio = mean_function([1.3,1.5,1,1.75,1,1.24,1.2,1])
      if "rye" in act["name"]:
        residue_ratio = mean_function([1.75,1.7])
      if "maize" in act["name"]:
        residue_ratio = mean_function([1,1,0.9,2,1.3,1,0.7,1,1,1])
      if "sunflower" in act["name"]:
        residue_ratio = mean_function([1.5,2.6,1.4,])
      if "rape" in act["name"]:
        residue_ratio = mean_function([1.1,1.7,1.7])
      if "rice" in act["name"]:
        residue_ratio = mean_function([1.76,1])

      unused_biomass_amount = 1 * residue_ratio * (1- mean_moisture_content_resdidue)

    wanted_act = act

    new_exc = wanted_act.new_exchange(input= biomass_unused, amount=unused_biomass_amount, type = "biosphere")
    new_exc["name"] = "Biomass, unused"
    new_exc.save()
    #print(wanted_act)

### Forestry ###

for act in bd.Database(ecoinvent_name):
  skip = True
  if "market" not in act["name"]:
    for cat in forestry_categories_list:
      if cat in act["classifications"][0]:
        for ex in act.exchanges():
          if ex["name"] == "Biomass, used":
            skip = False
            used_biomass_amount = ex.amount
            residue_ratio = 4 / 6

  for ex in act.exchanges():
    if ex["name"] == "Biomass, unused":
      skip = True

  if skip == False:
    unused_biomass_amount = used_biomass_amount * residue_ratio

    wanted_act = act
    new_exc = wanted_act.new_exchange(input= biomass_unused, amount=unused_biomass_amount, type = "biosphere")
    new_exc["name"] = "Biomass, unused"
    new_exc.save()
    #print(wanted_act)

### Animal ###
act = wanted_act
for act in bd.Database(ecoinvent_name):
  skip = True
  if "market" not in act["name"]:
    for cat in forestry_categories_list:
      if cat in act["classifications"][0]:
        for ex in act.exchanges():
          if ex["name"] == "Biomass, used":
            skip = False
            used_biomass_amount = ex.amount
            residue_ratio = 0

  for ex in act.exchanges():
    if ex["name"] == "Biomass, unused":
      skip = True

  if skip == False:
    unused_biomass_amount = used_biomass_amount * residue_ratio

    wanted_act = act
    new_exc = wanted_act.new_exchange(input= biomass_unused, amount=unused_biomass_amount, type = "biosphere")
    new_exc["name"] = "Biomass, unused"
    new_exc.save()
    #print(wanted_act)

### Fishing ###


for act in bd.Database(ecoinvent_name):
  skip = True
  if "market" not in act["name"]:
    if act["classifications"][0] == "0311:Marine fishing":
      for ex in act.exchanges():
        if ex["name"] == "Biomass, used":
          skip = False
          used_biomass_amount = ex.amount
          residue_ratio = 4 / 6

  for ex in act.exchanges():
    if ex["name"] == "Biomass, unused":
      skip = True

  if skip == False:
    for ex in act.exchanges():
      if ex["name"] == "Biomass, used":
        used_biomass_amount = ex.amount

    unused_biomass_amount = used_biomass_amount * residue_ratio

    wanted_act = act
    new_exc = wanted_act.new_exchange(input= biomass_unused, amount=unused_biomass_amount, type = "biosphere")
    new_exc["name"] = "Biomass, unused"
    new_exc.save()
    #print(wanted_act)



In [ ]:
##################
## MI Method  ####
##################

abiotic_rmi = []
abiotic_tmr = []
biotic_rmi  = []
biotic_tmr  = []




for x in bd.Database(biosphere_name):

  # abiotic resources
  if x['categories'][0] == 'natural resource' and x['categories'][1] == 'in ground' and x["unit"] == "kilogram" and "Overburden" not in x['name']:
    abiotic_rmi.append((x,1))
    abiotic_tmr.append((x,1))
  if x['categories'][0] == 'natural resource' and x['categories'][1] == 'in ground' and x["unit"] == "standard cubic meter":
    abiotic_rmi.append((x,0.8))
    abiotic_tmr.append((x,0.8))


  if x['name'] == "Overburden" :
    abiotic_tmr.append((x,1))


  # biotic resources
  if x["name"] == "Biomass, used":
    biotic_rmi.append((x,1))
    biotic_tmr.append((x,1))

  if x["name"] == "Biomass, unused":
    biotic_tmr.append((x,1))


mi_abiotic_rmi_method_key = ('PMF (direct) Abiotic RMI', 'imaginaryendpoint', 'imaginarymidpoint')
bd.Method(mi_abiotic_rmi_method_key).register()
bd.Method(mi_abiotic_rmi_method_key).write(abiotic_rmi)
bd.Method(mi_abiotic_rmi_method_key).load()

mi_abiotic_tmr_method_key = ('PMF (direct) Abiotic TMR', 'imaginaryendpoint', 'imaginarymidpoint')
bd.Method(mi_abiotic_tmr_method_key).register()
bd.Method(mi_abiotic_tmr_method_key).write(abiotic_tmr)
bd.Method(mi_abiotic_tmr_method_key).load()

mi_biotic_rmi_method_key = ('PMF (direct) Biotic RMI', 'imaginaryendpoint', 'imaginarymidpoint')
bd.Method(mi_biotic_rmi_method_key).register()
bd.Method(mi_biotic_rmi_method_key).write(biotic_rmi)
bd.Method(mi_biotic_rmi_method_key).load()

mi_biotic_tmr_method_key = ('PMF (direct) Biotic TMR', 'imaginaryendpoint', 'imaginarymidpoint')
bd.Method(mi_biotic_tmr_method_key).register()
bd.Method(mi_biotic_tmr_method_key).write(biotic_tmr)
bd.Method(mi_biotic_tmr_method_key).load()


[(205626474705764352, 1), (205626474563158016, 1)]

# Test PMF Method

In [ ]:
"""
proc_name         = "steel production, electric, chromium steel 18/8"
reference_product = "steel, chromium steel 18/8"
location          = "RER"

wanted_act = [act for act in bd.Database(ecoinvent_name) if proc_name in act['name']
              and reference_product.strip() in act['reference product']
              and location in act['location']][0]

rmi_abiotic_key = [m for m in bd.methods if "PMF (direct) Abiotic RMI" in str(m)][0]

fu = {wanted_act:1}

lca = bc.LCA(fu,rmi_abiotic_key)
lca.lci()
lca.lcia()
LCA_Ergebnis = lca.score

print("abiotic RMI for " + wanted_act["name"] + "  :  " + str(LCA_Ergebnis) + " kg")
"""

abiotic RMI for steel production, electric, chromium steel 18/8  :  22.823791696637166 kg


In [ ]:
"""
ca = ba.ContributionAnalysis()

# Now call the method on the instance
top_emm = ca.annotated_top_emissions(lca, limit=10, names=True)

for score, amount, flow in top_emm:
    print(f"{flow}: {score} (amount: {amount})")
"""

'Gangue' (kilogram, None, ('natural resource', 'in ground')): 17.73419555166614 (amount: 17.734195551666144)
'Coal, hard' (kilogram, None, ('natural resource', 'in ground')): 1.2836878185468146 (amount: 1.2836878185468141)
'Gravel' (kilogram, None, ('natural resource', 'in ground')): 1.150492242587961 (amount: 1.1504922425879607)
'Iron' (kilogram, None, ('natural resource', 'in ground')): 0.5855502025099819 (amount: 0.585550202509982)
'Calcite' (kilogram, None, ('natural resource', 'in ground')): 0.4017628235960137 (amount: 0.4017628235960136)
'Gas, natural' (standard cubic meter, None, ('natural resource', 'in ground')): 0.33401157761634953 (amount: 0.41751446579898627)
'Shale' (kilogram, None, ('natural resource', 'in ground')): 0.2853242200824373 (amount: 0.2853242200824374)
'Coal, brown' (kilogram, None, ('natural resource', 'in ground')): 0.2705699660906512 (amount: 0.27056996609065104)
'Chromium' (kilogram, None, ('natural resource', 'in ground')): 0.20683167409033687 (amount: 0.